### ReAct Agent Architecture

#### Aim
This is the intuition behind ReAct, a general agent architecture.

1. act - let the model call specific tools
2. observe - pass the tool output back to the model
3. reason - let the model reason about the tool output to decide what to do next (e.g., call another tool or just respond directly)


In [5]:
import arxiv as arxiv_lib
import wikipedia
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

In [2]:
# 1. Define a patch function that acts as the missing .results() method
def legacy_results_patch(self):
    client = arxiv_lib.Client()
    return client.results(self)

# 2. Inject this method directly into the arxiv.Search class
arxiv_lib.Search.results = legacy_results_patch

# 3. Initialize your LangChain tools normally
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [3]:
arxiv.invoke("Attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifi'

In [6]:
# 1. Clear the library's internal query cache to wipe out old failures
wikipedia.wikipedia.search._cache.clear()

# 2. Force the global API endpoint to use HTTPS securely
wikipedia.wikipedia.API_URL = 'https://www.wikipedia.org'

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [7]:
wiki.invoke("Attention is all you need")

'Page: Attention Is All You Need\nSummary: "Attention Is All You Need" is a 2017 research paper in machine learning authored by eight scientists and engineers working at Google. The paper introduced a new deep learning architecture known as the transformer, based on the attention mechanism proposed in 2014 by Bahdanau et al. The transformer approach it describes has become the main architecture of a wide variety of artificial intelligence, including large language models. At the time, the focus of'

In [9]:
from dotenv import load_dotenv
load_dotenv()

import os

os.environ["TAVILY_API_KEY"]= os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]= os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]="ReAct-agent"

In [10]:
## Custom Functions
def multiply(a: int, b: int) -> int:
    """Multiply a and b

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Adds a and b
    
    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> int:
    """Divides a by b
    
    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [arxiv, wiki, multiply, add, divide]



In [11]:
## Tavily Search Tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()

/var/folders/ch/3xtnt0vx13q1n9pmf7ks_nhc0000gn/T/ipykernel_82001/1622265974.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults()


In [14]:
tavily.invoke("Provide me the recent AI news for june 1st 2026")

[{'title': 'EP 589 | Daily AI News | June 1, 2026: 500 Million Data Points. AI Found the Answer.',
  'url': 'https://johnsviokla.substack.com/p/ep-589-daily-ai-news-june-1-2026',
  'content': "Rating: Optional  \nRationale: This article argues that Generative AI will affect SaaS unevenly depending on data type and whether outputs are deterministic or predictive. Our analysts found the framework useful, but not actionable enough for AI leaders deciding whether to replace or retain major SaaS platforms.\n\nMistral AI Launches Vibe, Expands into Industrial AI And Announces Data Center Push to Challenge OpenAI\n\nRating: Optional  \nRationale: Mistral AI is expanding with Vibe, industrial AI capabilities, and a data center strategy aimed at enterprise and sovereign AI needs. Our analysts noted the strategic importance of verticalization, Europe-focused AI infrastructure, and open-weight models, but rated it optional for most AI leaders outside those contexts. [...] AI Is Already Rewiring t